In [7]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_tavily import TavilySearch
from langchain.tools import tool
import requests 


In [4]:
from langchain.agents import create_agent

In [3]:
load_dotenv()
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"]=os.getenv("TAVILY_API_KEY")
os.environ["WHEATHERSTACK_API_KEY"]=os.getenv("WHEATHERSTACk_API_KEY")

In [8]:
search_tool = TavilySearch(max_results=3)

In [9]:
@tool
def get_weather(city: str) -> str:
    """Fetch current weather information for a city"""

    api_key = os.getenv("WEATHERSTACK_API_KEY")

    if not api_key:
        return "WEATHERSTACK_API_KEY is not set"

    url = (
        f"http://api.weatherstack.com/current"
        f"?access_key={api_key}"
        f"&query={city}"
    )

    response = requests.get(url)
    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}: {data}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [10]:
result= search_tool.invoke("What is the latest news on AI?")
result

{'query': 'What is the latest news on AI?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.artificialintelligence-news.com',
   'title': 'AI News | Latest News | Insights Powering AI-Driven Business Growth',
   'content': 'October 30, 2025\n\n### Malaysia launches Ryt Bank, its first AI-powered bank\n\nFinance AI\n\nAugust 26, 2025\n\n### Google’s Veo 3 AI video creation tools are now widely available\n\nAI in Action\n\nJuly 29, 2025\n\n#### Computer Vision\n\n### Microsoft’s Majorana 2 quantum chip is also a case study for agentic AI in R&D\n\nInside AI\n\nJune 3, 2026\n\n### US and Japan announce sweeping AI and tech collaboration\n\nArtificial Intelligence\n\nApril 11, 2024\n\n### UK and Canada sign AI compute agreement [...] August 18, 2026\n\n### Okta targets AI agent token costs with MCP scoping\n\nData Engineering & MLOps\n\nAugust 13, 2026\n\n### Red Hat, NVIDIA, IBM back project turning AI policy into code\n\nGovernance, Regulat

In [12]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=os.environ["GROQ_API_KEY"]
)

In [14]:
response = llm.invoke("Hello, explain AI in one sentence")
response

AIMessage(content='Artificial intelligence is the field of computer science that creates systems capable of performing tasks that normally require human intelligence, such as learning, reasoning, and perception.', additional_kwargs={'reasoning_content': 'User wants "Explain AI in one sentence". We must comply. It\'s a short definition. Provide a concise sentence. Ensure no policy violations. Provide a single sentence.'}, response_metadata={'token_usage': {'completion_tokens': 73, 'prompt_tokens': 78, 'total_tokens': 151, 'completion_time': 0.121014784, 'completion_tokens_details': {'reasoning_tokens': 34}, 'prompt_time': 0.004629467, 'prompt_tokens_details': None, 'queue_time': 0.219906451, 'total_time': 0.125644251}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_a8c584dda7', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a07b26-419d-7602-bb00-c2e1efbfecb5-0', tool_calls=[], invalid_tool_calls=[], usage_m

In [16]:
tools=[search_tool,get_weather]

In [18]:
agent =create_agent(
    model=llm,
    tools=tools
)

In [21]:
response = agent.invoke({
    "messages": [
        {
            "role": "user","content": "Find the current weather in Ongole"
        }
    ]
})

response

{'messages': [HumanMessage(content='Find the current weather in Ongole', additional_kwargs={}, response_metadata={}, id='502b002a-5d41-4c4c-a2d2-04edec761011'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to get current weather in Ongole. Use get_weather function.', 'tool_calls': [{'id': 'fc_dcdf05b9-1828-45dc-b427-dcf204856050', 'function': {'arguments': '{"city":"Ongole"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 343, 'total_tokens': 384, 'completion_time': 0.042484398, 'completion_tokens_details': {'reasoning_tokens': 16}, 'prompt_time': 0.018281924, 'prompt_tokens_details': None, 'queue_time': 0.344038165, 'total_time': 0.060766322}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_84bb35977d', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a07b2b-d1e1-7e52-849f-6b6ca9baa1a5-0', tool_cal

In [23]:
print(response["messages"][-1].content)

**Current weather in Ongole (as of 02:00 PM, 07 September 2026)**  
| Parameter | Value |
|-----------|-------|
| Condition | Overcast |
| Temperature | **38 °C** (Feels like 39 °C) |
| Humidity | 33 % |
| Wind | 11.5 km/h from the East (92°) |
| UV Index | 3.1 (Moderate) |
| Air pressure | 1005 mb |
| Cloud cover | 91 % |
| Visibility | 10 km |
| Precipitation | 0 mm (17 % chance of rain) |

**Source** – *Ongole Weather Conditions* (AQI.in) – “Here is a complete overview of Ongole's current weather conditions as of 02:00 PM 07 September 2026.” (score 0.954)  

If you need a forecast for the next few days or more detailed data, let me know!


In [25]:
for msg in response["messages"]:
    print(msg.content)

Find the current weather in Ongole

WEATHERSTACK_API_KEY is not set

{"query": "current weather in Ongole", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.aqi.in/weather/india/andhra-pradesh/ongole", "title": "Ongole Weather Conditions: Temperature | 30 Days Forecast", "content": "Here is a complete overview of Ongole's current weather conditions as of 02:00 PM 07 September 2026:\nCondition : Overcast\nTemperature : 38°C Hot (Feels like 39°C)\nHumidity : 33%\nWind Speed : 11.5 km/h from E (92 degrees), Gust 3.8 m/s\nUV Index : 3.1 (Moderate\nAir Pressure : 1005 mb\nCloud Cover : 91%\nVisibility : 10 km\nPrecipitation : 0 mm (17% chance of rain)\n\n3. What is the 10-day weather forecast for Ongole from 02:00 PM 07 September 2026? [...] India flag\n\n38 °C\n\nsunnySunny\n\nsunny\n\nHumidity Icon28%\n\nHumidity Icon\n\nUV Icon5.3\n\nUV Icon\n\n10.8 kmph /  NNW\n\nLast Updated:  07 Sept 2026, 08:57 AM\n\n## FAQs About Ongole Weather Conditions\n